# SurviveCity v1 — extended Kaggle T4 retrain

The original Colab run was **12 GRPO steps in 3h53m**, which gave a clean
1.7× reward / 2.0× episode-length lift but only 2 logged training points
(steps 10 and 12). This notebook **trains from base for 20 steps** and saves
every 4, so the next eval pass yields:

  - a 5-point training curve (steps 4, 8, 12, 16, 20) instead of 2 points
  - a cross-checkpoint eval trend (the curve in §5.3 of the report)
  - tighter confidence intervals via larger `n_t` / `n_b`

Time budget on Kaggle T4: ~19 min/step × 20 = **~6.3 hours of training** plus
~30 min eval. Fits well inside Kaggle's 12h session cap.

**Repo isolation: this notebook pushes to `noanya/zombiee-v1-extended`,
not `noanya/zombiee`.** The original step-12 adapter, checkpoints, TB run dir,
and eval JSON on `noanya/zombiee` are NOT modified by this run. If the new
numbers turn out worse or broken, fall back to the original — no recovery
needed. If they're better, repoint v1.tex's URLs to the new repo and ship.


## 0. Bind to a single GPU BEFORE any imports

Kaggle ships dual T4 by default; bnb + DataParallel breaks CUBLAS in that
combo. Setting CUDA_VISIBLE_DEVICES=0 here (before torch is imported anywhere)
makes torch see exactly one device.


In [2]:
import os, sys, time
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("python:", sys.version.split()[0])
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])


python: 3.12.12
CUDA_VISIBLE_DEVICES: 0


## 1. Pull HF + GitHub tokens (Kaggle Secrets → env)

**HF_TOKEN** is required (the Hub push will fail without it).
**GITHUB_TOKEN** is only required if the GitHub repo is private; public clone
works without it.


In [4]:
HF_TOKEN = None
GITHUB_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
    try: HF_TOKEN = _sec.get_secret("HF_TOKEN")
    except Exception: pass
    try: GITHUB_TOKEN = _sec.get_secret("GITHUB_TOKEN")
    except Exception: pass
except ImportError:
    pass
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
GITHUB_TOKEN = GITHUB_TOKEN or os.environ.get("GITHUB_TOKEN")
assert HF_TOKEN, "Set HF_TOKEN in Kaggle Secrets (Add-ons → Secrets) before running."
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACE_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
print(f"HF_TOKEN     set ({len(HF_TOKEN)} chars, prefix={HF_TOKEN[:6]}...)")
print(f"GITHUB_TOKEN {'set ('+str(len(GITHUB_TOKEN))+' chars)' if GITHUB_TOKEN else 'absent (ok if repo is public)'}")


HF_TOKEN     set (37 chars, prefix=hf_jrZ...)
GITHUB_TOKEN set (40 chars)


## 2. Pin torch cu121 FIRST so accelerate can't downgrade it

In [5]:
import subprocess
def pip(*args, check=True):
    cmd = [sys.executable, "-m", "pip"] + list(args)
    print(">>", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout: print(r.stdout[-1500:])
    if r.stderr: print("STDERR:", r.stderr[-800:])
    if check and r.returncode != 0: raise RuntimeError(f"pip failed (rc={r.returncode})")
    return r

if "torch" in sys.modules:
    raise RuntimeError("torch already imported — restart kernel via Run → Factory reset.")

pip("install", "-q",
    "--index-url", "https://download.pytorch.org/whl/cu121",
    "--upgrade-strategy=only-if-needed",
    "torch==2.5.1+cu121", "torchvision==0.20.1+cu121", "torchaudio==2.5.1+cu121")


>> /usr/bin/python3 -m pip install -q --index-url https://download.pytorch.org/whl/cu121 --upgrade-strategy=only-if-needed torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 108.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 97.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 70.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 49.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 87.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 32.6 MB/s eta 0:00:00
 

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '--index-url', 'https://download.pytorch.org/whl/cu121', '--upgrade-strategy=only-if-needed', 'torch==2.5.1+cu121', 'torchvision==0.20.1+cu121', 'torchaudio==2.5.1+cu121'], returncode=0, stdout='     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.3 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 108.1 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 97.2 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 70.7 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 49.7 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 87.5 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.2 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 1.8 MB/s eta 0:00:0

## 3. Install pinned dep stack (no-deps + force-reinstall)

`--no-deps + --force-reinstall` guarantees our pins take effect even on
Kaggle's pre-populated dist-packages (the Achilles heel of the v2 run).
The base image already has compatible tokenizers/safetensors/numpy.


In [6]:
# All packages that transformers 4.46.3 has range constraints on go in the
# strict pin block. Kaggle ships latest-of-everything pre-installed, so any
# unpinned dep above transformers' upper bound will crash dependency_versions_check.
pip("install", "-q", "--no-deps", "--force-reinstall",
    "transformers==4.46.3",
    "tokenizers==0.20.3",       # transformers 4.46 needs >=0.20,<0.21
    "huggingface_hub==0.27.1",  # transformers 4.46 needs >=0.23.2,<1.0
    "peft==0.13.2",
    "trl==0.15.2",
    "bitsandbytes==0.43.3",
    "accelerate==1.1.1",
    "datasets==3.1.0")
# Do NOT install mergekit. trl 0.15+ touches it via callbacks.py, but only
# imports the actual mergekit submodules if `is_mergekit_available()` returns
# True. With mergekit absent, that check returns False, the conditional
# `from mergekit.common import ...` is skipped, and GRPO works fine — we
# never call any merge utilities. Installing it with `--no-deps` was the
# original mistake here: mergekit needs `immutables` (and friends) at runtime,
# and `--no-deps` skipped them, so the import succeeded at metadata level
# but crashed when trl actually loaded mergekit.common during GRPOTrainer
# construction.
pip("uninstall", "-y", "mergekit", check=False)
pip("install", "-q", "tensorboard", "tbparse", "psutil", "pydantic")
pip("uninstall", "-y", "torchao", check=False)

# transformers 4.46 removed TRANSFORMERS_CACHE; some optional imports still use it.
# Purge any half-imported transformers from a previous failed run before re-import.
import sys
for _m in [m for m in sys.modules if m.startswith("transformers") or m.startswith("huggingface_hub")]:
    del sys.modules[_m]
import transformers.utils.hub as _hub
if not hasattr(_hub, "TRANSFORMERS_CACHE"):
    _hub.TRANSFORMERS_CACHE = getattr(_hub, "HF_HUB_CACHE", None) or os.path.expanduser("~/.cache/huggingface/hub")


>> /usr/bin/python3 -m pip install -q --no-deps --force-reinstall transformers==4.46.3 tokenizers==0.20.3 huggingface_hub==0.27.1 peft==0.13.2 trl==0.15.2 bitsandbytes==0.43.3 accelerate==1.1.1 datasets==3.1.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.7/450.7 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 38.0 MB/s eta 0:00:00

>> /usr/bin/python3 -m pip uninstall -y mergekit
STDERR: WARNING: S

## 4. Verify pin took + torch sees the GPU

In [7]:
import importlib.metadata as _m
expected = {"transformers":"4.46.3","tokenizers":"0.20.3",
            "huggingface_hub":"0.27.1",
            "peft":"0.13.2","trl":"0.15.2","bitsandbytes":"0.43.3",
            "accelerate":"1.1.1","datasets":"3.1.0"}
bad = []
for pkg, want in expected.items():
    try:
        got = _m.version(pkg)
        ok = got == want
        print(f"  {pkg:15s} want={want:10s} got={got:10s} {'OK' if ok else 'MISMATCH'}")
        if not ok: bad.append((pkg, want, got))
    except _m.PackageNotFoundError:
        print(f"  {pkg:15s} NOT INSTALLED"); bad.append((pkg, want, "missing"))
assert not bad, f"pinned versions didn't take: {bad}"

import torch
print(f"\n  torch={torch.__version__}")
print(f"  cuda available: {torch.cuda.is_available()}  device count: {torch.cuda.device_count()}")
print(f"  torch built with cuda: {torch.version.cuda}")
print(f"  CUDA_VISIBLE_DEVICES env: {os.environ.get('CUDA_VISIBLE_DEVICES','(unset)')}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA not available. Most common causes on Kaggle:\n"
        "  1) Notebook's accelerator is OFF. Fix: Settings (right sidebar) → "
        "Accelerator → 'GPU T4 x1' or 'GPU P100', then restart kernel.\n"
        "  2) torch.version.cuda is None → CPU-only torch was installed. "
        "Re-run cell 2 (the cu121 pin) and ensure cell 3 didn't pull a CPU torch.\n"
        f"     This kernel reports torch.version.cuda = {torch.version.cuda!r}.\n"
        "  3) Weekly GPU quota exhausted on this Kaggle account. Check the "
        "'Your Account' page for remaining hours."
    )
print(f"  GPU: {torch.cuda.get_device_name(0)} cc={torch.cuda.get_device_capability(0)}")
free, total = torch.cuda.mem_get_info(0)
print(f"  VRAM free={free/1e9:.1f}/{total/1e9:.1f} GB")

from transformers.utils.import_utils import is_bitsandbytes_available
assert is_bitsandbytes_available(), "bnb compat broken; check the pin install logs above."
print("  bnb compat OK")

# Final dep-check: actually IMPORT the heavy modules AND the names we'll
# use. transformers' dependency_versions_check.py and trl's lazy-loaded
# submodules both raise ImportError if any range constraint or transitive
# dep is missing — catching that here is much cheaper than catching it 30
# min into training.
print("\n  importing transformers, peft, trl + GRPOTrainer (the real dep check)...")
import transformers, peft, trl
from trl import GRPOTrainer, GRPOConfig  # forces trl.trainer.callbacks +
                                          # trl.mergekit_utils to load
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
print(f"    transformers {transformers.__version__}  peft {peft.__version__}  trl {trl.__version__}")
print("  full dep stack imports cleanly")

# Pre-flight: Kaggle internet check. Without this, the model download in the
# training cell crashes with a confusing network error 3 minutes in.
print("\n  checking Kaggle internet (HF Hub reachable)...")
import urllib.request, urllib.error
try:
    urllib.request.urlopen("https://huggingface.co/api/models/Qwen/Qwen2.5-3B-Instruct",
                           timeout=10).read(100)
    print("  internet OK — Hub is reachable")
except (urllib.error.URLError, TimeoutError) as _e:
    raise RuntimeError(
        f"Cannot reach huggingface.co: {_e}\n"
        f"Fix: Kaggle Settings (right sidebar) → Internet → 'On'. "
        f"Restart kernel after toggling and re-run from cell 0."
    )


  transformers    want=4.46.3     got=4.46.3     OK
  tokenizers      want=0.20.3     got=0.20.3     OK
  huggingface_hub want=0.27.1     got=0.27.1     OK
  peft            want=0.13.2     got=0.13.2     OK
  trl             want=0.15.2     got=0.15.2     OK
  bitsandbytes    want=0.43.3     got=0.43.3     OK
  accelerate      want=1.1.1      got=1.1.1      OK
  datasets        want=3.1.0      got=3.1.0      OK

  torch=2.5.1+cu121
  cuda available: True  device count: 1
  torch built with cuda: 12.1
  CUDA_VISIBLE_DEVICES env: 0
  GPU: Tesla T4 cc=(7, 5)
  VRAM free=15.5/15.6 GB
  bnb compat OK

  importing transformers, peft, trl + GRPOTrainer (the real dep check)...


2026-04-25 21:57:58.112419: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777154278.357783      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777154278.409513      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777154278.919297      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777154278.919347      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777154278.919350      55 computation_placer.cc:177] computation placer alr

    transformers 4.46.3  peft 0.13.2  trl 0.15.2
  full dep stack imports cleanly

  checking Kaggle internet (HF Hub reachable)...
  internet OK — Hub is reachable


## 5. Clone repo + cd into v1

Uses `GIT_TERMINAL_PROMPT=0` + `GIT_ASKPASS=/bin/echo` so a missing token
fails fast instead of hanging waiting for a username. If the GitHub repo is
private, `GITHUB_TOKEN` (from cell 1) is embedded via `http.extraheader` —
this avoids baking the token into the cloned repo's `.git/config`.

If clone still fails, the most likely cause is the repo being private
without GITHUB_TOKEN set in Kaggle Secrets, or a typo in the URL.


In [8]:
import base64
os.environ["GIT_TERMINAL_PROMPT"] = "0"
os.environ["GIT_ASKPASS"] = "/bin/echo"
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/tmp"
REPO_DIR = os.path.join(WORK, "zombiee")
REPO_URL = os.environ.get("REPO_URL", "https://github.com/SirjanSingh/zombiee.git")

if os.path.isdir(REPO_DIR):
    print(f"repo already cloned at {REPO_DIR}")
    if GITHUB_TOKEN:
        # Authenticated pull. Without the token the pull's stderr leaks an
        # "Authentication failed" message (still non-fatal — clone is intact —
        # but visually alarming).
        auth = "Authorization: Basic " + base64.b64encode(
            f"x-access-token:{GITHUB_TOKEN}".encode()
        ).decode()
        r = subprocess.run(
            ["git", "-c", f"http.extraheader={auth}", "-C", REPO_DIR,
             "pull", "--ff-only"],
            capture_output=True, text=True,
        )
        print("  pull:", r.stdout.strip() or r.stderr.strip()[:200] or "(no change)")
    else:
        print("  skipping pull (no GITHUB_TOKEN; existing checkout is fine).")
else:
    cmd = ["git", "clone", "--depth=1", REPO_URL, REPO_DIR]
    if GITHUB_TOKEN:
        # http.extraheader keeps the token out of the resulting .git/config
        auth = "Authorization: Basic " + base64.b64encode(
            f"x-access-token:{GITHUB_TOKEN}".encode()
        ).decode()
        cmd = ["git", "-c", f"http.extraheader={auth}"] + cmd[1:]
        print(f"cloning {REPO_URL} (with GITHUB_TOKEN auth)...")
    else:
        print(f"cloning {REPO_URL} (no GITHUB_TOKEN; only works if repo is public)...")
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout); print(r.stderr)
    if r.returncode != 0:
        raise RuntimeError(
            f"git clone failed (rc={r.returncode}). Likely causes:\n"
            f"  1) Repo is private and GITHUB_TOKEN isn't set. Add it to "
            f"Kaggle Secrets (Add-ons → Secrets) with key 'GITHUB_TOKEN'.\n"
            f"  2) Typo in REPO_URL ({REPO_URL}). Override via env var.\n"
            f"  3) Token doesn't have repo read access. Generate a new "
            f"fine-grained token with 'Contents: Read' on the target repo."
        )
V1 = os.path.join(REPO_DIR, "v1")
assert os.path.isdir(V1), f"v1/ not found at {V1} (repo may have a different layout)"
os.chdir(V1)
sys.path.insert(0, V1)
print("CWD:", os.getcwd())


cloning https://github.com/SirjanSingh/zombiee.git (with GITHUB_TOKEN auth)...

Cloning into '/kaggle/working/zombiee'...

CWD: /kaggle/working/zombiee/v1


## 6. Pre-create / login to the Hub repo

`HUB_REPO` is INTENTIONALLY a separate repo from `noanya/zombiee`. The
original step-12 artefacts must remain untouched — they're what the report's
existing figures and Table 2 are derived from. If this extended run lands
cleanly, repoint `report/v1/v1.tex`'s URLs to the new repo. If it doesn't,
nothing on `noanya/zombiee` changed; just abandon this run.


In [9]:
# Override via env var if you want a different name; the default is
# isolated from the original `noanya/zombiee`.
HUB_REPO = os.environ.get("HUB_REPO", "noanya/zombiee-v1-extended")
ORIGINAL_REPO = "noanya/zombiee"
assert HUB_REPO != ORIGINAL_REPO, (
    f"HUB_REPO must NOT equal {ORIGINAL_REPO} — that would overwrite the "
    f"step-12 root adapter. Pick a different name (default is "
    f"'noanya/zombiee-v1-extended')."
)
print(f"  training will push to: https://huggingface.co/{HUB_REPO}")
print(f"  original repo (untouched): https://huggingface.co/{ORIGINAL_REPO}")

from huggingface_hub import HfApi, login
login(token=HF_TOKEN, add_to_git_credential=False)
HfApi().create_repo(repo_id=HUB_REPO, repo_type="model", exist_ok=True, private=False)
print(f"  hub repo ready.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


  training will push to: https://huggingface.co/noanya/zombiee-v1-extended
  original repo (untouched): https://huggingface.co/noanya/zombiee
  hub repo ready.


## 7. Train — 20 steps from base, save every 4

Hyperparameters (the only departures from the original Colab run are noted):

  - `--max-steps 20`         (was 12; the whole point of this run)
  - `--save-steps 4`         (5 checkpoints: 4, 8, 12, 16, 20)
  - `--save-total-limit 5`
  - `--num-generations 2`    (was 4; paired with per_device=2 below — TRL
                              0.15+ requires global_batch_size to be divisible
                              by num_generations, and v1's hardcoded
                              per_device=1 broke that constraint. Total
                              compute per gradient step is unchanged:
                              per_device(2) × grad_accum(16) × num_gen(2)
                              = 64 evaluations, same as v1's original
                              1 × 16 × 4 = 64. Group size 2 is GRPO's
                              minimum but still gives a usable signal.)
  - `--lr 1e-5`              (unchanged)
  - `--temperature 0.9`      (unchanged)
  - `--beta 0.04`            (unchanged)
  - `--lora-r 16`            (unchanged)

The cell below also inline-patches v1/training/train.py to set
`per_device_train_batch_size=2` (the only change beyond the CLI args).

Expected: ~19 min/step × 20 = ~6.3h on T4. Hub pushes happen on every save.


In [10]:
# v1's train.py hardcodes per_device_train_batch_size=1, but TRL 0.15+
# requires (num_processes × per_device_batch) to be divisible by
# num_generations. Single GPU + batch=1 + num_gen=4 fails. Patch to
# per_device=2 paired with num_gen=2 — same total compute and same
# per-micro-batch memory as the original v1 Colab run.
import re, pathlib as _pl
_tp = _pl.Path("training/train.py")
_src = _tp.read_text()
if "per_device_train_batch_size=2," not in _src:
    _new = re.sub(
        r"per_device_train_batch_size=1,",
        "per_device_train_batch_size=2,  # patched for TRL 0.15+ num_gen constraint",
        _src, count=1,
    )
    if _new == _src:
        raise RuntimeError("could not patch per_device_train_batch_size in train.py")
    _tp.write_text(_new)
    print("patched train.py: per_device_train_batch_size 1 → 2")
else:
    print("train.py already patched")

import importlib, training.train as _train_mod
importlib.reload(_train_mod)

ARGS = [
    "--model-name", "Qwen/Qwen2.5-3B-Instruct",
    "--max-steps", "20",
    "--save-steps", "4",
    "--save-total-limit", "5",
    "--num-generations", "2",   # paired with per_device=2 (TRL 0.15+ needs global%num_gen==0)
    "--lr", "1e-5",
    "--temperature", "0.9",
    "--beta", "0.04",
    "--lora-r", "16",
    "--lora-alpha", "32",
    "--seed", "42",
    "--output-dir", os.path.join(WORK, "lora_v1_extend"),
    "--push-to-hub",
    "--hub-model-id", HUB_REPO,
]
sys.argv = ["train.py"] + ARGS
print("Launching training:")
for a in ARGS: print("  ", a)
print()
_train_mod.main()


2026-04-25 21:58:39,466 INFO GPU=Tesla T4 cc=7.5 cuda=12.1 bf16=False fp16=True


patched train.py: per_device_train_batch_size 1 → 2
Launching training:
   --model-name
   Qwen/Qwen2.5-3B-Instruct
   --max-steps
   20
   --save-steps
   4
   --save-total-limit
   5
   --num-generations
   2
   --lr
   1e-5
   --temperature
   0.9
   --beta
   0.04
   --lora-r
   16
   --lora-alpha
   32
   --seed
   42
   --output-dir
   /kaggle/working/lora_v1_extend
   --push-to-hub
   --hub-model-id
   noanya/zombiee-v1-extended



tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

2026-04-25 21:59:14,968 INFO GPU memory after load: free=12.12GB total=15.64GB
2026-04-25 21:59:14,988 INFO [START] episode=1 infected=A2 seed=670487
2026-04-25 21:59:14,990 INFO [START] episode=1 infected=A2 seed=116739
2026-04-25 21:59:14,991 INFO [START] episode=1 infected=A2 seed=26225
2026-04-25 21:59:14,991 INFO [START] episode=1 infected=A2 seed=777572
2026-04-25 21:59:14,992 INFO [START] episode=1 infected=A0 seed=288389
2026-04-25 21:59:14,993 INFO [START] episode=1 infected=A1 seed=256787
2026-04-25 21:59:14,994 INFO [START] episode=1 infected=A0 seed=234053
2026-04-25 21:59:14,995 INFO [START] episode=1 infected=A2 seed=146316
2026-04-25 21:59:14,996 INFO [START] episode=1 infected=A0 seed=772246
2026-04-25 21:59:14,997 INFO [START] episode=1 infected=A2 seed=107473
2026-04-25 21:59:14,998 INFO [START] episode=1 infected=A2 seed=709570
2026-04-25 21:59:14,999 INFO [START] episode=1 infected=A2 seed=776646
2026-04-25 21:59:15,000 INFO [START] episode=1 infected=A1 seed=935518

Step,Training Loss
10,0.000000
20,0.000000


2026-04-25 22:12:22,304 INFO [START] episode=1 infected=A0 seed=360663
2026-04-25 22:12:22,305 INFO [STEP] ep=1 step=0 agent=A0 action=eat reward=0.0100 raw=0.0050 done=False
2026-04-25 22:12:22,306 INFO [STEP] ep=1 step=0 agent=A1 action=move_down reward=0.0100 raw=0.0050 done=False
2026-04-25 22:12:22,307 INFO [STEP] ep=1 step=1 agent=A2 action=move_up reward=0.0100 raw=0.0050 done=False
2026-04-25 22:12:22,309 INFO [STEP] ep=1 step=1 agent=A0 action=move_up reward=0.0100 raw=0.0050 done=False
2026-04-25 22:12:22,310 INFO [STEP] ep=1 step=1 agent=A1 action=move_right reward=0.0100 raw=0.0050 done=False
2026-04-25 22:12:22,311 INFO [STEP] ep=1 step=2 agent=A2 action=eat reward=0.0100 raw=0.0050 done=False
2026-04-25 22:12:22,312 INFO [STEP] ep=1 step=2 agent=A0 action=move_up reward=0.0100 raw=0.0050 done=False
2026-04-25 22:12:22,313 INFO [STEP] ep=1 step=2 agent=A1 action=wait reward=0.0100 raw=0.0050 done=False
2026-04-25 22:12:22,314 INFO [STEP] ep=1 step=3 agent=A2 action=move_le

## 8. Training summary — TB curve across the 5 logged points

In [13]:
from tbparse import SummaryReader
import pandas as pd, glob

ckpt_dir = os.path.join(WORK, "lora_v1_extend")
runs = sorted(glob.glob(os.path.join(ckpt_dir, "runs", "*")))
for r in runs[-2:]:
    print(f"\n=== {os.path.basename(r)} ===")
    df = SummaryReader(r, pivot=True).scalars
    if df is None or df.empty:
        print("  (empty)"); continue
    df = df.sort_values("step").reset_index(drop=True)
    cols = ["step"] + [c for c in [
        "train/loss", "train/reward", "train/reward_std", "train/kl",
        "train/grad_norm", "train/learning_rate"
    ] if c in df.columns]
    print(df[cols].to_string(index=False))



=== Apr25_21-59-15_b81378f6e0fe ===
 step  train/loss  train/reward  train/reward_std  train/kl  train/grad_norm  train/learning_rate
   10         0.0          0.01               0.0       0.0              0.0             0.000005
   20         0.0          0.01               0.0       0.0              0.0             0.000000
